# AEGIS Brain Tumor Segmentation (Glioma) - Revised
**Adaptive Engine for Governance and Integrity Surveillance**

This notebook implements the AEGIS framework for trustworthy AI governance, adapted for a Glioma Segmentation use case. This revised version directly addresses feedback from peer review, specifically:

1.  **Fixing the \"Rigid Non-Regression\" Rule (Reviewer 3, Major #3):** A tolerance (τ = 0.015) has been added to the performance regression check. A model is now only rejected if its performance drops significantly below the released version, preventing the system from getting locked into a `REJECT` state due to minor, clinically insignificant fluctuations.
2.  **Generating a Quantitative Results Table (Reviewer 2):** This notebook produces a clear, iteration-by-iteration summary of the governance decisions, which can be directly used in the manuscript.

The simulation reads pre-computed metric logs (JSON files) to demonstrate the governance process.


In [12]:
# Section 1 — Setup, Configuration & Metric Loading
# This section defines the core parameters and helper functions for the simulation. 
# The `DecisionConfig` class now includes the `regression_tolerance` as suggested by the reviewers. 
# The `get_metrics_from_file` function is responsible for loading and parsing the JSON log files from each iteration.

import os
import json
import glob
import numpy as np
import pandas as pd
import warnings
from dataclasses import dataclass

# --- Configuration ---
LOG_PATH = "./metrics_logs"
MAX_ITERATIONS = 20  # Set to a high number to process all available files

@dataclass
class DecisionConfig:
    """
    Thresholds for AEGIS governance, updated to include reviewer feedback.
    """
    target_region: str = "tc"
    target_metric: str = "dice" # dice, nsd, or hd95
    performance_floor: float = 0.676
    
    # REVIEWER FIX: Added a tolerance to the non-regression rule.
    regression_tolerance: float = 0.015

def get_metrics_from_file(json_path):
    """Loads and computes mean metrics from a JSON file."""
    if not os.path.exists(json_path):
        return None

    with open(json_path, "r") as file:
        json_data = json.load(file)

    metrics_data = json_data.get("metrics", [])
    if not metrics_data:
        return {}

    collected_metrics = {}
    for subject_data in metrics_data:
        for key, region_data in subject_data.items():
            if key == "subject_name":
                continue
            if key not in collected_metrics:
                collected_metrics[key] = {'dice': [], 'nsd': [], 'hd95': []}

            if isinstance(region_data, dict):
                # Extract values and replace inf with nan
                dsc = region_data.get("global_bin_dsc", np.nan)
                nsd = region_data.get("global_bin_nsd", np.nan)
                hd95 = region_data.get("sq_hd95", np.nan)

                collected_metrics[key]['dice'].append(dsc if np.isfinite(dsc) else np.nan)
                collected_metrics[key]['nsd'].append(nsd if np.isfinite(nsd) else np.nan)
                collected_metrics[key]['hd95'].append(hd95 if np.isfinite(hd95) else np.nan)
            else:
                collected_metrics[key]['dice'].append(np.nan)
                collected_metrics[key]['nsd'].append(np.nan)
                collected_metrics[key]['hd95'].append(np.nan)

    stats_results = {}
    for region, data in collected_metrics.items():
        region_key_lower = region.lower()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            for metric_name in ['dice', 'nsd', 'hd95']:
                scores = data[metric_name]
                stats_results[f"{metric_name}_{region_key_lower}"] = np.nanmean(scores) if not np.all(np.isnan(scores)) else 0
            
    return stats_results


## Section 2 — Conditional Decision Module (CDM)

This module contains the core governance logic. It has been updated to reflect the binary **APPROVE/REJECT** system and the new **Performance Regression** rule with a tolerance margin.


In [13]:
class ConditionalDecisionModule:
    """AEGIS CDM adapted for Glioma Segmentation based on the paper's logic."""

    def __init__(self, config: DecisionConfig):
        self.config = config
        self.released_golden_metric = None
        self.performance_floor = None
        
        # HD95 is a distance metric, so a lower score is better.
        # Dice and NSD are overlap metrics, so a higher score is better.
        self.higher_is_better = (self.config.target_metric.lower() != "hd95")

    def set_performance_floor(self, metrics: dict):
        """Calculates the safety floor based on the Golden benchmark (Iteration 0)."""
        metric_key = f"{self.config.target_metric}_{self.config.target_region}"
        baseline_val = metrics.get(metric_key, 0.0)

        if self.higher_is_better:
            self.performance_floor = baseline_val - 0.05
        else:
            self.performance_floor = baseline_val + 0.05
            
    def set_initial_release_performance(self, metrics: dict):
        """Stores the baseline metric score from the first approved model."""
        metric_key = f"{self.config.target_metric}_{self.config.target_region}"
        self.released_golden_metric = metrics.get(metric_key)

    def update_release_performance(self, metrics: dict):
        """Updates the stored performance to the newly approved model's metrics."""
        metric_key = f"{self.config.target_metric}_{self.config.target_region}"
        self.released_golden_metric = metrics.get(metric_key)

    def evaluate(self, current_golden_metrics: dict, current_drifting_metrics: dict, prev_drifting_metrics: dict):
        """Evaluates the metrics against the updated governance protocol for Glioma."""
        pccp = self._evaluate_pccp(current_golden_metrics)
        alarm, alarm_reasons = self._evaluate_pms_alarm(current_drifting_metrics, prev_drifting_metrics)

        return {
            "decision": pccp["label"],
            "trigger_reasons": pccp["reasons"],
            "pms_alarm": alarm,
            "pms_alarm_reasons": alarm_reasons,
        }

    def _evaluate_pccp(self, golden_metrics: dict):
        """PCCP Decision Chain for Glioma: Binary Approve/Reject with tolerance."""
        c = self.config
        metric_key = f"{c.target_metric}_{c.target_region}"
        current_metric = golden_metrics.get(metric_key, 0.0)
        metric_upper = c.target_metric.upper()
        region_upper = c.target_region.upper()

        # 1. REJECT: Safety Violation Check
        if self.performance_floor is not None:
            if self.higher_is_better:
                if current_metric < self.performance_floor:
                    return self._label("REJECT", [f"Metric {current_metric:.3f} < floor {self.performance_floor:.3f} (Safety Violation)"])
            else: # Lower is better (HD95)
                if current_metric > self.performance_floor:
                    return self._label("REJECT", [f"Metric {current_metric:.3f} > floor {self.performance_floor:.3f} (Safety Violation)"])

        # 2. REJECT: Performance Regression Check
        if self.released_golden_metric is not None:
            if self.higher_is_better:
                # REVIEWER FIX: Reject only if performance drops by more than the tolerance.
                if current_metric < self.released_golden_metric - c.regression_tolerance:
                    return self._label("REJECT", [f"Metric {current_metric:.3f} shows regression vs. released {self.released_golden_metric:.3f} (Tolerance: {c.regression_tolerance})"])
            else: # Lower is better
                if current_metric > self.released_golden_metric + c.regression_tolerance:
                    return self._label("REJECT", [f"Metric {current_metric:.3f} shows regression vs. released {self.released_golden_metric:.3f} (Tolerance: {c.regression_tolerance})"])

        # 3. APPROVE: If no rejection criteria are met
        return self._label("APPROVE", ["Performance meets or exceeds criteria"])

    def _evaluate_pms_alarm(self, current_drifting_metrics: dict, prev_drifting_metrics: dict):
        """PMS ALARM signal based on performance drop on drifting data."""
        if prev_drifting_metrics is None or current_drifting_metrics is None:
            return False, []

        c = self.config
        metric_key = f"{c.target_metric}_{c.target_region}"
        prev_metric = prev_drifting_metrics.get(metric_key, 0.0)
        current_metric = current_drifting_metrics.get(metric_key, 0.0)

        if self.higher_is_better:
            if current_metric < prev_metric:
                return True, [f"ALARM: Performance drop on new data ({c.target_metric.upper()} {prev_metric:.3f} -> {current_metric:.3f})"]
        else: # Lower is better
            if current_metric > prev_metric:
                return True, [f"ALARM: Performance drop on new data ({c.target_metric.upper()} {prev_metric:.3f} -> {current_metric:.3f})"]
            
        return False, []

    def _label(self, label: str, reasons: list):
        return {"label": label, "reasons": reasons}


## Section 3 — Iterative Simulation

This section processes each iteration of the pre-computed logs, applying the AEGIS governance logic, and records the outcomes.


In [14]:
target_region = "tc"
target_metric = "dice"

print("=" * 80)
print(f"AEGIS Glioma Segmentation Governance Simulation (Region: {target_region.upper()}, Metric: {target_metric.upper()})")
print("=" * 80)

config = DecisionConfig(target_region=target_region.lower(), target_metric=target_metric.lower())
cdm = ConditionalDecisionModule(config)
results_log = []
previous_drifting_metrics = None
all_golden_metrics = {}
all_drifting_metrics = {}

# --- Iteration 0: Establish Gold Standard ---
# Iteration 0 is only used to set the baseline for the ALARM check. It is not a "released" model.
print("\\n--- Iteration 0: Baseline ---")
baseline_pattern = os.path.join(LOG_PATH, "iteration0_*_golden.json")
baseline_matches = glob.glob(baseline_pattern)

if not baseline_matches:
    print(f"Error: Baseline metrics file not found. Aborting.")
else:
    baseline_path = baseline_matches[0]
    golden_metrics_0 = get_metrics_from_file(baseline_path)

    if golden_metrics_0 is not None:
        # Save for later use in ablation study
        all_golden_metrics[0] = golden_metrics_0
        
        # Dynamically set performance floor based on Baseline
        metric_key = f"{target_metric.lower()}_{target_region.lower()}"
        baseline_val = golden_metrics_0.get(metric_key, 0.0)
        cdm.set_performance_floor(golden_metrics_0)
        config.performance_floor = cdm.performance_floor
        
        print(f"Baseline {target_metric.upper()} ({target_region.upper()}) is: {baseline_val:.3f}")
        print(f"Performance Floor set to: {cdm.performance_floor:.3f} (based on Iteration 0 - 0.05)")
        print("Note: Iteration 0 is NOT a released model. The first released model will be determined from Iteration 1.")

        # Load drift data for iteration 0 to use in the next iteration's alarm check
        drift0_pattern = os.path.join(LOG_PATH, "iteration0_*_drift.json")
        drift0_matches = glob.glob(drift0_pattern)
        if drift0_matches:
            previous_drifting_metrics = get_metrics_from_file(drift0_matches[0])
            all_drifting_metrics[0] = previous_drifting_metrics

        # --- Iterations 1 to MAX_ITERATIONS ---
        for i in range(1, MAX_ITERATIONS):
            golden_pattern = os.path.join(LOG_PATH, f"iteration{i}_*_golden.json")
            drift_pattern = os.path.join(LOG_PATH, f"iteration{i}_*_drift.json")

            golden_matches = glob.glob(golden_pattern)
            if not golden_matches:
                continue

            golden_path = golden_matches[0]
            current_golden_metrics = get_metrics_from_file(golden_path)
            all_golden_metrics[i] = current_golden_metrics
            
            current_drifting_metrics = None
            drift_matches = glob.glob(drift_pattern)
            if drift_matches:
                current_drifting_metrics = get_metrics_from_file(drift_matches[0])
                all_drifting_metrics[i] = current_drifting_metrics

            if current_golden_metrics is None:
                continue

            # The first iteration is compared against no released model yet.
            decision = cdm.evaluate(current_golden_metrics, current_drifting_metrics, previous_drifting_metrics)
            
            # If this is the first approval, set the initial released performance.
            if cdm.released_golden_metric is None and decision["decision"] == "APPROVE":
                cdm.set_initial_release_performance(current_golden_metrics)
                print(f"\\n--- Iteration {i}: First Model Approved ---")
                print(f"Initial Released Performance set to: {cdm.released_golden_metric:.3f}")
            elif decision["decision"] == "APPROVE":
                cdm.update_release_performance(current_golden_metrics)

            metric_key = f"{target_metric.lower()}_{target_region.lower()}"
            results_log.append({
                "Iter": i,
                f"{target_metric.upper()} ({target_region.upper()})": f"{current_golden_metrics.get(metric_key, 0):.3f}",
                "Drift Perf.": f"{current_drifting_metrics.get(metric_key, 0):.3f}" if current_drifting_metrics else "N/A",
                "PCCP Decision": decision["decision"],
                "PMS ALARM": "Yes" if decision["pms_alarm"] else "No",
                "Trigger": "; ".join(decision["trigger_reasons"] + decision["pms_alarm_reasons"]),
            })
            
            previous_drifting_metrics = current_drifting_metrics


AEGIS Glioma Segmentation Governance Simulation (Region: TC, Metric: DICE)
\n--- Iteration 0: Baseline ---
Baseline DICE (TC) is: 0.726
Performance Floor set to: 0.676 (based on Iteration 0 - 0.05)
Note: Iteration 0 is NOT a released model. The first released model will be determined from Iteration 1.
\n--- Iteration 3: First Model Approved ---
Initial Released Performance set to: 0.690


## Section 4 — Governance Summary

This final section prints the complete, iteration-by-iteration governance report. This table provides the quantitative results requested by the reviewers and demonstrates the effect of the updated, more robust decision logic.


In [15]:
if results_log:
    summary_df = pd.DataFrame(results_log)
    print("\\n" + "=" * 100)
    print(f"AEGIS Glioma Segmentation Governance Summary (Region: {target_region.upper()}, Metric: {target_metric.upper()})")
    print("=" * 100)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.max_colwidth', None):
        print(summary_df)


\n====================================================================================================
AEGIS Glioma Segmentation Governance Summary (Region: TC, Metric: DICE)
    Iter DICE (TC) Drift Perf. PCCP Decision PMS ALARM                                                                                                                         Trigger
0      1     0.646       0.694        REJECT        No                                                                                   Metric 0.646 < floor 0.676 (Safety Violation)
1      2     0.673       0.748        REJECT        No                                                                                   Metric 0.673 < floor 0.676 (Safety Violation)
2      3     0.690       0.745       APPROVE       Yes                                Performance meets or exceeds criteria; ALARM: Performance drop on new data (DICE 0.748 -> 0.745)
3      4     0.700       0.758       APPROVE        No                                       

## Section 5 — Ablation Studies

This section explores the sensitivity of the governance logic to the chosen hyperparameters.

We perform two ablation studies:
1.  **Performance Floor:** Evaluates the effect of changing the safety margin used to establish the performance floor from Iteration 0 baseline performance.
2.  **Regression Tolerance:** Evaluates the impact of modifying the tolerance margin (τ) allowed before a model is rejected due to performance degradation compared to the previously released model.


### 5.1 Ablation Study: Performance Floor Margin
The safety performance floor is defined as `baseline_val - margin`. We vary this `margin` from `0.01` to `0.10` with a step of `0.01` and observe how it affects the sequence of APPROVE/REJECT decisions.


In [16]:
print("\\n" + "=" * 80)
print("Ablation Study 5.1: Performance Floor Margin")
print("=" * 80)

floor_margins = np.round(np.arange(0.01, 0.11, 0.01), 2)
ablation_floor_results = []
metric_key = f"{target_metric.lower()}_{target_region.lower()}"

baseline_val = all_golden_metrics[0].get(metric_key, 0.0)

for margin in floor_margins:
    # Initialize a new config and CDM for each ablation step
    abl_config = DecisionConfig(target_region=target_region.lower(), target_metric=target_metric.lower())
    abl_cdm = ConditionalDecisionModule(abl_config)
    abl_cdm.performance_floor = baseline_val - margin
    
    decisions_for_margin = {"Margin": f"{margin:.2f}", "Floor": f"{abl_cdm.performance_floor:.3f}"}
    
    for i in range(1, MAX_ITERATIONS):
        if i not in all_golden_metrics:
            continue
            
        current_golden_metrics = all_golden_metrics[i]
        
        # We only care about PCCP decisions here, so we skip drifting metrics
        pccp = abl_cdm._evaluate_pccp(current_golden_metrics)
        decision = pccp["label"]
        
        if decision == "APPROVE":
            if abl_cdm.released_golden_metric is None:
                abl_cdm.set_initial_release_performance(current_golden_metrics)
            else:
                abl_cdm.update_release_performance(current_golden_metrics)
                
        decisions_for_margin[f"Iter {i}"] = decision
        
    ablation_floor_results.append(decisions_for_margin)

ablation_floor_df = pd.DataFrame(ablation_floor_results)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(ablation_floor_df.to_string(index=False))


\n================================================================================
Ablation Study 5.1: Performance Floor Margin
Margin Floor  Iter 1  Iter 2  Iter 3  Iter 4  Iter 5  Iter 6  Iter 7 Iter 8 Iter 9 Iter 10 Iter 11 Iter 12
  0.01 0.716  REJECT  REJECT  REJECT  REJECT APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.02 0.706  REJECT  REJECT  REJECT  REJECT APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.03 0.696  REJECT  REJECT  REJECT APPROVE APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.04 0.686  REJECT  REJECT APPROVE APPROVE APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.05 0.676  REJECT  REJECT APPROVE APPROVE APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.06 0.666  REJECT APPROVE APPROVE APPROVE APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.07 0.656  REJECT APPROVE APPROVE APPROVE APPROVE APPROVE APPROVE REJECT REJECT  REJECT  REJECT  REJECT
  0.08 0.646  RE

### 5.2 Ablation Study: Regression Tolerance (τ)
A new model is rejected if its performance drops below the last released model by more than the `regression_tolerance` (τ). We vary this tolerance from `0.010` to `0.020` with a step of `0.001` to assess its effect on deployment decisions.


In [17]:
print("\\n" + "=" * 80)
print("Ablation Study 5.2: Regression Tolerance (τ)")
print("=" * 80)

tolerance_values = np.round(np.arange(0.010, 0.021, 0.001), 3)
ablation_tol_results = []

# Using default margin of 0.05 for this study
default_floor = baseline_val - 0.05

for tol in tolerance_values:
    # Initialize a new config and CDM for each ablation step
    abl_config = DecisionConfig(target_region=target_region.lower(), target_metric=target_metric.lower())
    abl_config.regression_tolerance = tol
    abl_cdm = ConditionalDecisionModule(abl_config)
    abl_cdm.performance_floor = default_floor
    
    decisions_for_tol = {"Tolerance": f"{tol:.3f}"}
    
    for i in range(1, MAX_ITERATIONS):
        if i not in all_golden_metrics:
            continue
            
        current_golden_metrics = all_golden_metrics[i]
        
        # We only care about PCCP decisions here
        pccp = abl_cdm._evaluate_pccp(current_golden_metrics)
        decision = pccp["label"]
        
        if decision == "APPROVE":
            if abl_cdm.released_golden_metric is None:
                abl_cdm.set_initial_release_performance(current_golden_metrics)
            else:
                abl_cdm.update_release_performance(current_golden_metrics)
                
        decisions_for_tol[f"Iter {i}"] = decision
        
    ablation_tol_results.append(decisions_for_tol)

ablation_tol_df = pd.DataFrame(ablation_tol_results)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(ablation_tol_df.to_string(index=False))


\n================================================================================
Ablation Study 5.2: Regression Tolerance (τ)
Tolerance Iter 1 Iter 2  Iter 3  Iter 4  Iter 5  Iter 6  Iter 7  Iter 8  Iter 9 Iter 10 Iter 11 Iter 12
    0.010 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.011 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.012 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.013 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.014 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.015 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.016 REJECT REJECT APPROVE APPROVE APPROVE APPROVE APPROVE  REJECT  REJECT  REJECT  REJECT  REJECT
    0.017 REJECT REJECT APPROVE APPROVE 

## Section 6 — Framework Compatibility Simulation


**Note:** This is a *framework compatibility test* to show that the system *can* support these intermediate states, even though they were deemed clinically inappropriate for the final, high-consequence brain tumor governance protocol. In fact, the `Clinical Review` and `Conditional Approval` categories were deactivated for the glioma instantiation because brain tumor segmentation is a high-stakes task used for radiotherapy and surgical planning. In this safety-critical context, intermediate states carry severe risks and lack clinical utility, restricting deployment decisions strictly to a binary APPROVE or REJECT outcome. This intentional asymmetry demonstrates the flexibility of the AEGIS framework, proving that manufacturers can customize active categories to match a device's specific clinical risk profile

This section demonstrates the adaptability of the AEGIS framework by simulating the 4-level decision logic (including `CLINICAL_REVIEW` and `CONDITIONAL_APPROVAL`) from the Sepsis example on the Glioma data.

In [18]:
@dataclass
class DecisionConfig4Level(DecisionConfig):
    """Extends the config to include thresholds for the 4-level Sepsis logic."""
    review_dice: float = 0.88
    minor_drift_low: float = 0.30
    minor_drift_high: float = 0.70
    major_drift_threshold: float = 0.90

class FourLevelCDM(ConditionalDecisionModule):
    """A CDM that implements the 4-level logic for compatibility testing."""
    
    def _evaluate_pccp(self, golden_metrics: dict, drift_score: float, pms_alarm: bool):
        c = self.config
        metric_key = f"{c.target_metric}_{c.target_region}"
        current_metric = golden_metrics.get(metric_key, 0.0)
        metric_upper = c.target_metric.upper()
        region_upper = c.target_region.upper()

        # P1: REJECT
        if current_metric < self.performance_floor:
            return self._label("REJECT", [f"Metric {current_metric:.3f} < floor {self.performance_floor:.3f} (Safety Violation)"])

        # P2: CLINICAL_REVIEW
        if hasattr(c, 'review_dice') and current_metric < c.review_dice:
            return self._label("CLINICAL_REVIEW", [f"Metric {current_metric:.3f} < review threshold {c.review_dice:.3f}"])

        # P3: CONDITIONAL_APPROVAL
        if hasattr(c, 'minor_drift_low') and c.minor_drift_low <= drift_score <= c.minor_drift_high:
            return self._label("CONDITIONAL_APPROVAL", [f"Minor drift detected: score={drift_score:.3f}"])

        # P4: APPROVE (with Gold Standard check)
        if not pms_alarm and self.released_golden_metric is not None:
            if self.higher_is_better:
                if current_metric < self.released_golden_metric - c.regression_tolerance:
                    return self._label("CLINICAL_REVIEW", [f"Metric regression vs. released"])
            else:
                if current_metric > self.released_golden_metric + c.regression_tolerance:
                    return self._label("CLINICAL_REVIEW", [f"Metric regression vs. released"])
        
        reasons = ["All criteria satisfied"]
        if pms_alarm:
            reasons.append("Gold standard check deferred")
        return self._label("APPROVE", reasons)

    def evaluate(self, current_golden_metrics: dict, current_drifting_metrics: dict, prev_drifting_metrics: dict, drift_data: dict):
        """Overridden evaluate to include drift score in PCCP logic."""
        drift_score = drift_data.get("drift_score", 0.0)
        
        # PMS ALARM evaluation remains the same (can be based on performance or drift)
        alarm, alarm_reasons = self._evaluate_pms_alarm(current_drifting_metrics, prev_drifting_metrics)
        
        # Pass drift score to PCCP evaluation
        pccp = self._evaluate_pccp(current_golden_metrics, drift_score, pms_alarm=alarm)

        return {
            "decision": pccp["label"],
            "trigger_reasons": pccp["reasons"],
            "pms_alarm": alarm,
            "pms_alarm_reasons": alarm_reasons,
        }

# --- Run Simulation ---
print("\\n" + "=" * 80)
print("Framework Compatibility Test: 4-Level Decision Logic")
print("=" * 80)

config_4_level = DecisionConfig4Level()
cdm_4_level = FourLevelCDM(config_4_level)
cdm_4_level.performance_floor = cdm.performance_floor # Carry over the floor from the main simulation
compat_results_log = []

for i in range(1, MAX_ITERATIONS):
    if i not in all_golden_metrics:
        continue
        
    current_golden = all_golden_metrics.get(i)
    current_drift_perf = all_drifting_metrics.get(i)
    prev_drift_perf = all_drifting_metrics.get(i-1)
    
    # The 4-level logic requires a drift score, which we don't have in the glioma logs.
    # We will simulate a drift score for demonstration purposes.
    # Let's create a synthetic drift score that triggers different states.
    simulated_drift_score = 0.0
    if i == 6: simulated_drift_score = 0.45  # Triggers CONDITIONAL_APPROVAL
    if i == 8: simulated_drift_score = 0.95  # Triggers PMS ALARM
    
    simulated_drift_data = {"drift_score": simulated_drift_score}

    decision = cdm_4_level.evaluate(current_golden, current_drift_perf, prev_drift_perf, simulated_drift_data)
    
    if cdm_4_level.released_golden_metric is None and decision["decision"] == "APPROVE":
        cdm_4_level.set_initial_release_performance(current_golden)
    elif decision["decision"] == "APPROVE":
        cdm_4_level.update_release_performance(current_golden)
        
    compat_results_log.append({
        "Iter": i,
        f"Dice ({target_region.upper()})": f"{current_golden.get(metric_key, 0):.3f}",
        "Sim. Drift": f"{simulated_drift_score:.3f}",
        "PCCP Decision": decision["decision"],
        "PMS ALARM": "Yes" if decision["pms_alarm"] else "No",
        "Trigger": "; ".join(decision["trigger_reasons"] + decision["pms_alarm_reasons"]),
    })

if compat_results_log:
    compat_df = pd.DataFrame(compat_results_log)
    print("\\n" + "=" * 100)
    print("AEGIS Framework Compatibility Summary (4-Level Logic)")
    print("=" * 100)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.max_colwidth', None):
        print(compat_df)


\n================================================================================
Framework Compatibility Test: 4-Level Decision Logic
\n====================================================================================================
AEGIS Framework Compatibility Summary (4-Level Logic)
    Iter Dice (TC) Sim. Drift    PCCP Decision PMS ALARM                                                                                           Trigger
0      1     0.646      0.000           REJECT        No                                                     Metric 0.646 < floor 0.676 (Safety Violation)
1      2     0.673      0.000           REJECT        No                                                     Metric 0.673 < floor 0.676 (Safety Violation)
2      3     0.690      0.000  CLINICAL_REVIEW       Yes  Metric 0.690 < review threshold 0.880; ALARM: Performance drop on new data (DICE 0.748 -> 0.745)
3      4     0.700      0.000  CLINICAL_REVIEW        No                               

## Section 7 - Assessment over other tumor subregion
The governance pipeline in AEGIS is highly modular and adaptable. The analysis can easily be performed by choosing different tumor subregions, such as Enhancing Tumor ("et"), Non-Enhancing Tumor Core ("netc"), or Whole Tumor ("wt").

Below, we showcase how the governance changes if we switch the tumor subregion from Tumor Core ("tc") to Whole Tumor ("wt").


In [19]:
target_region = "wt"
target_metric = "dice"

print("=" * 80)
print(f"AEGIS Glioma Segmentation Governance Simulation (Region: {target_region.upper()}, Metric: {target_metric.upper()})")
print("=" * 80)

config = DecisionConfig(target_region=target_region.lower(), target_metric=target_metric.lower())
cdm = ConditionalDecisionModule(config)
results_log = []
previous_drifting_metrics = None

# --- Iteration 0: Establish Gold Standard ---
print("\\n--- Iteration 0: Baseline ---")
baseline_pattern = os.path.join(LOG_PATH, "iteration0_*_golden.json")
baseline_matches = glob.glob(baseline_pattern)

if not baseline_matches:
    print(f"Error: Baseline metrics file not found. Aborting.")
else:
    baseline_path = baseline_matches[0]
    golden_metrics_0 = get_metrics_from_file(baseline_path)

    if golden_metrics_0 is not None:
        # Dynamically set performance floor based on Baseline
        metric_key = f"{target_metric.lower()}_{target_region.lower()}"
        baseline_val = golden_metrics_0.get(metric_key, 0.0)
        cdm.set_performance_floor(golden_metrics_0)
        config.performance_floor = cdm.performance_floor
        
        print(f"Baseline {target_metric.upper()} ({target_region.upper()}) is: {baseline_val:.3f}")
        print(f"Performance Floor set to: {cdm.performance_floor:.3f} (based on Iteration 0 - 0.05)")
        print("Note: Iteration 0 is NOT a released model. The first released model will be determined from Iteration 1.")

        # Load drift data for iteration 0 to use in the next iteration's alarm check
        drift0_pattern = os.path.join(LOG_PATH, "iteration0_*_drift.json")
        drift0_matches = glob.glob(drift0_pattern)
        if drift0_matches:
            previous_drifting_metrics = get_metrics_from_file(drift0_matches[0])

        # --- Iterations 1 to MAX_ITERATIONS ---
        for i in range(1, MAX_ITERATIONS):
            golden_pattern = os.path.join(LOG_PATH, f"iteration{i}_*_golden.json")
            drift_pattern = os.path.join(LOG_PATH, f"iteration{i}_*_drift.json")

            golden_matches = glob.glob(golden_pattern)
            if not golden_matches:
                continue

            golden_path = golden_matches[0]
            current_golden_metrics = get_metrics_from_file(golden_path)
            
            current_drifting_metrics = None
            drift_matches = glob.glob(drift_pattern)
            if drift_matches:
                current_drifting_metrics = get_metrics_from_file(drift_matches[0])

            if current_golden_metrics is None:
                continue

            # The first iteration is compared against no released model yet.
            decision = cdm.evaluate(current_golden_metrics, current_drifting_metrics, previous_drifting_metrics)
            
            # Update the released model's performance if the decision is APPROVE
            if cdm.released_golden_metric is None and decision["decision"] == "APPROVE":
                cdm.set_initial_release_performance(current_golden_metrics)
                print(f"\\n--- Iteration {i}: First Model Approved ---")
                print(f"Initial Released Performance set to: {cdm.released_golden_metric:.3f}")
            elif decision["decision"] == "APPROVE":
                cdm.update_release_performance(current_golden_metrics)

            metric_key = f"{target_metric.lower()}_{target_region.lower()}"
            results_log.append({
                "Iter": i,
                f"{target_metric.upper()} ({target_region.upper()})": f"{current_golden_metrics.get(metric_key, 0):.3f}",
                "Drift Perf.": f"{current_drifting_metrics.get(metric_key, 0):.3f}" if current_drifting_metrics else "N/A",
                "PCCP Decision": decision["decision"],
                "PMS ALARM": "Yes" if decision["pms_alarm"] else "No",
                "Trigger": "; ".join(decision["trigger_reasons"] + decision["pms_alarm_reasons"]),
            })
            
            previous_drifting_metrics = current_drifting_metrics

if results_log:
    summary_df = pd.DataFrame(results_log)
    print("\\n" + "=" * 100)
    print(f"AEGIS Glioma Segmentation Governance Summary (Region: {target_region.upper()}, Metric: {target_metric.upper()})")
    print("=" * 100)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.max_colwidth', None):
        print(summary_df)


AEGIS Glioma Segmentation Governance Simulation (Region: WT, Metric: DICE)
\n--- Iteration 0: Baseline ---
Baseline DICE (WT) is: 0.894
Performance Floor set to: 0.844 (based on Iteration 0 - 0.05)
Note: Iteration 0 is NOT a released model. The first released model will be determined from Iteration 1.
\n--- Iteration 1: First Model Approved ---
Initial Released Performance set to: 0.867
\n====================================================================================================
AEGIS Glioma Segmentation Governance Summary (Region: WT, Metric: DICE)
    Iter DICE (WT) Drift Perf. PCCP Decision PMS ALARM                                                                                           Trigger
0      1     0.867       0.908       APPROVE        No                                                             Performance meets or exceeds criteria
1      2     0.880       0.981       APPROVE        No                                                             Performance me

Furthermore, the target metric can be easily adapted. Below we demonstrate the governance pipeline using the Normalized Surface Distance (NSD) for the Non-Enhancing Tumor Core(NETC) region.


In [21]:
target_region = "netc"
target_metric = "nsd"

print("=" * 80)
print(f"AEGIS Glioma Segmentation Governance Simulation (Region: {target_region.upper()}, Metric: {target_metric.upper()})")
print("=" * 80)

config = DecisionConfig(target_region=target_region.lower(), target_metric=target_metric.lower())
cdm = ConditionalDecisionModule(config)
results_log = []
previous_drifting_metrics = None

# --- Iteration 0: Establish Gold Standard ---
print("\\n--- Iteration 0: Baseline ---")
baseline_pattern = os.path.join(LOG_PATH, "iteration0_*_golden.json")
baseline_matches = glob.glob(baseline_pattern)

if not baseline_matches:
    print(f"Error: Baseline metrics file not found. Aborting.")
else:
    baseline_path = baseline_matches[0]
    golden_metrics_0 = get_metrics_from_file(baseline_path)

    if golden_metrics_0 is not None:
        # Dynamically set performance floor based on Baseline
        metric_key = f"{target_metric.lower()}_{target_region.lower()}"
        baseline_val = golden_metrics_0.get(metric_key, 0.0)
        cdm.set_performance_floor(golden_metrics_0)
        config.performance_floor = cdm.performance_floor
        
        print(f"Baseline {target_metric.upper()} ({target_region.upper()}) is: {baseline_val:.3f}")
        print(f"Performance Floor set to: {cdm.performance_floor:.3f} (based on Iteration 0 - 0.05)")
        print("Note: Iteration 0 is NOT a released model. The first released model will be determined from Iteration 1.")

        # Load drift data for iteration 0 to use in the next iteration's alarm check
        drift0_pattern = os.path.join(LOG_PATH, "iteration0_*_drift.json")
        drift0_matches = glob.glob(drift0_pattern)
        if drift0_matches:
            previous_drifting_metrics = get_metrics_from_file(drift0_matches[0])

        # --- Iterations 1 to MAX_ITERATIONS ---
        for i in range(1, MAX_ITERATIONS):
            golden_pattern = os.path.join(LOG_PATH, f"iteration{i}_*_golden.json")
            drift_pattern = os.path.join(LOG_PATH, f"iteration{i}_*_drift.json")

            golden_matches = glob.glob(golden_pattern)
            if not golden_matches:
                continue

            golden_path = golden_matches[0]
            current_golden_metrics = get_metrics_from_file(golden_path)
            
            current_drifting_metrics = None
            drift_matches = glob.glob(drift_pattern)
            if drift_matches:
                current_drifting_metrics = get_metrics_from_file(drift_matches[0])

            if current_golden_metrics is None:
                continue

            # The first iteration is compared against no released model yet.
            decision = cdm.evaluate(current_golden_metrics, current_drifting_metrics, previous_drifting_metrics)
            
            # Update the released model's performance if the decision is APPROVE
            if cdm.released_golden_metric is None and decision["decision"] == "APPROVE":
                cdm.set_initial_release_performance(current_golden_metrics)
                print(f"\\n--- Iteration {i}: First Model Approved ---")
                print(f"Initial Released Performance set to: {cdm.released_golden_metric:.3f}")
            elif decision["decision"] == "APPROVE":
                cdm.update_release_performance(current_golden_metrics)

            metric_key = f"{target_metric.lower()}_{target_region.lower()}"
            results_log.append({
                "Iter": i,
                f"{target_metric.upper()} ({target_region.upper()})": f"{current_golden_metrics.get(metric_key, 0):.3f}",
                "Drift Perf.": f"{current_drifting_metrics.get(metric_key, 0):.3f}" if current_drifting_metrics else "N/A",
                "PCCP Decision": decision["decision"],
                "PMS ALARM": "Yes" if decision["pms_alarm"] else "No",
                "Trigger": "; ".join(decision["trigger_reasons"] + decision["pms_alarm_reasons"]),
            })
            
            previous_drifting_metrics = current_drifting_metrics

if results_log:
    summary_df = pd.DataFrame(results_log)
    print("\\n" + "=" * 100)
    print(f"AEGIS Glioma Segmentation Governance Summary (Region: {target_region.upper()}, Metric: {target_metric.upper()})")
    print("=" * 100)
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000, 'display.max_colwidth', None):
        print(summary_df)


AEGIS Glioma Segmentation Governance Simulation (Region: NETC, Metric: NSD)
\n--- Iteration 0: Baseline ---
Baseline NSD (NETC) is: 0.664
Performance Floor set to: 0.614 (based on Iteration 0 - 0.05)
Note: Iteration 0 is NOT a released model. The first released model will be determined from Iteration 1.
\n--- Iteration 4: First Model Approved ---
Initial Released Performance set to: 0.621
\n====================================================================================================
AEGIS Glioma Segmentation Governance Summary (Region: NETC, Metric: NSD)
    Iter NSD (NETC) Drift Perf. PCCP Decision PMS ALARM                                                                                          Trigger
0      1      0.589       0.483        REJECT        No                                                    Metric 0.589 < floor 0.614 (Safety Violation)
1      2      0.603       0.527        REJECT        No                                                    Metric 0.603 < floo